# logit-thermo 探索 T-scan：温度ごとの崩れ方（CPU）

- **必要な secret：なし**（`HF_TOKEN` があればトークナイザーの取り込みが少し速くなるだけ）。**ランタイム：CPU（標準）**。見込みは 15〜30 分。
- 探索（予想なし）。A2-2・A2-5・A2-6 で保存した生成のトークンをデコードして、温度ごとに「結晶（写し）・電子不足（代名詞）・具体物（固有名詞）・共有結合（登場人物のつながり）・気体（ほかの記事に出ない単語）」の割合を数え、`A2_explore/tscan_summary.json` に書きます。**文章の中身は表示も保存もしません。**
- 計算のコード：`scripts/a2_2_collect.py` は Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_explore_tscan.py`（`71a78cf86dcbd9b9…`）は埋め込み。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo 探索 T-scan（温度ごとの崩れ方、25 回、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 T-scan（温度ごとの崩れ方、25 回、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore T-scan', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53'})
SRC22 = _s['SRC22']
SRCTS = '"""探索 T-scan：温度ごとの崩れ方（結晶・電子不足・具体物・共有結合・気体）。2026-10-29 夕方。確かめには使わない（予想なし、後知恵用）。\nCPU のノートブックで、scripts/a2_2_collect.py を実行したのと同じ名前空間で実行する（load_docs・split_doc・TEMPS を使う）。\n生成はしない。A2-2・A2-5・A2-6 で保存した生成のトークン（温度 15・サンプル 2・記事 40・512 歩）をデコードして数えるだけ。\n文章の中身は表示しない・保存しない（数だけ）。\n\n数えるもの（続き 512 トークンをデコードした英語の文で。単語は [A-Za-z][A-Za-z\'-]* で切る）：\n  copy4  ：4 トークンの完全な写しの割合（トークン列で。書き出しを含めて、それまでに出た 4 つ組）。結晶。\n  pron   ：代名詞・指示語（下の PRON）の、単語に占める割合。ルイス酸（電子不足）。\n  proper ：文の頭以外で大文字で始まる単語（I を除く）の割合。ルイス塩基（具体物）。\n  bind   ：proper のうち、書き出しかそれより前の続きに同じ綴りで出ていたものの割合。共有結合（登場人物のつながり）。\n  gas    ：その記事以外の 39 本の人の記事（全文）に一度も出ない単語（小文字にして）の割合。気体（ぐちゃぐちゃ）。\n人の続き（同じ記事の 512 トークン）でも同じように数え、基準の線にする。\n"""\nimport json, re\nimport numpy as np\n\nTS_RUNS = {\n    \'A2_2\': [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'qwen2.5-0.5b\', \'Qwen/Qwen2.5-0.5B\'), (\'qwen2.5-0.5b-inst\', \'Qwen/Qwen2.5-0.5B-Instruct\'),\n             (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\'), (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'), (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\'),\n             (\'smollm2-1.7b-inst\', \'HuggingFaceTB/SmolLM2-1.7B-Instruct\')],\n    \'A2_5\': [(n, \'EleutherAI/pythia-1b\') for n in [\'pythia-70m\', \'pythia-160m\', \'pythia-410m\', \'pythia-1b\', \'pythia-1.4b\', \'pythia-2.8b\',\n                                                   \'pythia-1b@step8000\', \'pythia-1b@step24000\', \'pythia-1b@step64000\']],\n    \'A2_6\': [(\'qwen2.5-0.5b\', \'Qwen/Qwen2.5-0.5B\'), (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\'), (\'qwen2.5-3b\', \'Qwen/Qwen2.5-3B\'),\n             (\'smollm2-135m\', \'HuggingFaceTB/SmolLM2-135M\'), (\'smollm2-360m\', \'HuggingFaceTB/SmolLM2-360M\'),\n             (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\'), (\'qwen3-0.6b\', \'Qwen/Qwen3-0.6B-Base\'), (\'qwen3-1.7b\', \'Qwen/Qwen3-1.7B-Base\'),\n             (\'qwen3-4b\', \'Qwen/Qwen3-4B-Base\')],\n}\nTS_KEYS = [\'copy4\', \'pron\', \'proper\', \'bind\', \'gas\']\nPRON = {\'he\', \'she\', \'it\', \'they\', \'him\', \'her\', \'them\', \'his\', \'hers\', \'its\', \'their\', \'theirs\',\n        \'this\', \'that\', \'these\', \'those\'}\nWORD = re.compile(r"[A-Za-z][A-Za-z\'\\-]*")\nSENT_END = re.compile(r\'[.!?]["\\\')\\]]*\\s*$\')\n\n\ndef copy4(prompt, cont):\n    seq = [int(t) for t in prompt] + [int(t) for t in cont]\n    P, seen, c = len(prompt), set(), 0\n    for i in range(3, len(seq)):\n        g = tuple(seq[i - 3:i + 1])\n        if i >= P and g in seen:\n            c += 1\n        seen.add(g)\n    return c / max(len(cont), 1)\n\n\ndef words_with_pos(text):\n    """→ [(単語, 文の頭か)]。文の頭 ＝ 文字列の最初の単語、または直前の単語との間に . ! ? がある単語。"""\n    out, last = [], 0\n    for m in WORD.finditer(text):\n        between = text[last:m.start()]\n        start = (not out) or bool(re.search(r\'[.!?]\', between)) or (\'\\n\' in between)\n        out.append((m.group(0), start))\n        last = m.end()\n    return out\n\n\ndef measures(prompt_ids, cont_ids, prompt_text, cont_text, vocab_other):\n    ws = words_with_pos(cont_text)\n    n = len(ws)\n    prev_words = set(w for w, _ in words_with_pos(prompt_text))\n    pron = sum(w.lower() in PRON for w, _ in ws)\n    proper_hits, bind_hits = 0, 0\n    for w, st in ws:\n        if not st and w[0].isupper() and w != \'I\':\n            proper_hits += 1\n            bind_hits += int(w in prev_words)\n        prev_words.add(w)\n    gas = sum(w.lower() not in vocab_other for w, _ in ws)\n    return dict(copy4=copy4(prompt_ids, cont_ids), pron=pron / n if n else np.nan, proper=proper_hits / n if n else np.nan,\n                bind=bind_hits / proper_hits if proper_hits else np.nan, gas=gas / n if n else np.nan, n_words=n)\n\n\ndef run_tscan(base, docs):\n    from transformers import AutoTokenizer\n    doc_words = [set(w.lower() for w in WORD.findall(d[\'text\'])) for d in docs]\n    res = {\'temps\': list(TEMPS), \'runs\': {}, \'human\': {}}\n    tok_cache = {}\n    for folder, runs in TS_RUNS.items():\n        for name, hf in runs:\n            if hf not in tok_cache:\n                tok_cache[hf] = AutoTokenizer.from_pretrained(hf)\n            tok = tok_cache[hf]\n            pairs = [split_doc(tok, d[\'text\']) for d in docs]\n            Z = np.load(f\'{base}/{folder}/{name}.npz\')\n            use = [int(u) for u in Z[\'use\']]\n            toks = Z[\'toks\']                                     # (nT, S, nD, L)\n            assert [float(t) for t in Z[\'temps\']] == list(TEMPS)\n            nT, S, nD, L = toks.shape\n            vocab_other = {i: set().union(*[doc_words[k] for k in range(len(docs)) if k != i]) for i in use}\n            ptxt = {i: tok.decode(pairs[i][0], skip_special_tokens=True) for i in use}\n            curve = {k: np.full((nT, S, nD), np.nan) for k in TS_KEYS}\n            for kT in range(nT):\n                for s in range(S):\n                    for j, i in enumerate(use):\n                        c = toks[kT, s, j]\n                        m = measures(pairs[i][0], c, ptxt[i], tok.decode(c, skip_special_tokens=True), vocab_other[i])\n                        for k in TS_KEYS:\n                            curve[k][kT, s, j] = m[k]\n            hk = hf + \'|\' + \',\'.join(map(str, use))\n            if hk not in res[\'human\']:\n                hm = {k: [] for k in TS_KEYS}\n                for i in use:\n                    m = measures(pairs[i][0], pairs[i][1], ptxt[i], tok.decode(pairs[i][1], skip_special_tokens=True), vocab_other[i])\n                    for k in TS_KEYS:\n                        hm[k].append(m[k])\n                res[\'human\'][hk] = {k: float(np.nanmean(v)) for k, v in hm.items()}\n            res[\'runs\'][f\'{folder}/{name}\'] = dict(human_key=hk, **{k: np.nanmean(curve[k], axis=(1, 2)).tolist() for k in TS_KEYS},\n                                                   **{k + \'_doc_sd\': np.nanstd(np.nanmean(curve[k], axis=1), axis=1).tolist() for k in TS_KEYS})\n    return res\n\n\ndef self_test_tscan():\n    ws = words_with_pos(\'Mr Smith went home. He saw Paris and Smith again.\\nThe end\')\n    assert [w for w, st in ws if st] == [\'Mr\', \'He\', \'The\'], ws\n    m = measures([1, 2, 3], [1, 2, 3, 4, 1, 2, 3, 4], \'Smith arrived.\', \'Then Smith met Jones. It was fun and this was odd.\',\n                 {\'then\', \'smith\', \'met\', \'it\', \'was\', \'fun\', \'and\', \'this\'})\n    # 単語 11：Then Smith met Jones It was fun and this was odd。代名詞 it・this ＝ 2。固有名詞（文頭以外）Smith・Jones ＝ 2、そのうち前に出た Smith ＝ 1\n    assert abs(m[\'pron\'] - 2 / 11) < 1e-9 and abs(m[\'proper\'] - 2 / 11) < 1e-9 and m[\'bind\'] == 0.5, m\n    assert abs(m[\'gas\'] - 2 / 11) < 1e-9, m                       # jones・odd が辞書にない\n    assert abs(m[\'copy4\'] - 1 / 8) < 1e-9, m                      # 続きの 8 歩のうち、それまでに出た 4 つ組（1 2 3 4 の 2 回目）は 1 歩\n    return \'OK\'\n'
assert hashlib.sha256(SRCTS.encode()).hexdigest() == '71a78cf86dcbd9b9e5bbb04de0dd21db124c832dd969d3a88b0050ccde041b83'
exec(SRC22)
exec(SRCTS)
print('自己テスト：', self_test_tscan())


In [ ]:
# セル3：記事（A2-2 と同じか確かめる）→ 25 回を数える → 保存（結果は表示しない）
import json
try:
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    r = run_tscan('/content/drive/MyDrive/logit-thermo', docs)
    r['a2_explore_tscan_sha256'] = '71a78cf86dcbd9b9e5bbb04de0dd21db124c832dd969d3a88b0050ccde041b83'
    json.dump(r, open('/content/drive/MyDrive/logit-thermo' + '/A2_explore/tscan_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 T-scanで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：温度ごとの崩れ方を A2_explore/tscan_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
